# QASPER Result Consolidation

Public portfolio edition prepared for GitHub and Databricks. Credentials are read from environment variables; research data and generated artifacts are not committed to Git.


In [ ]:
# Databricks uses Unity Catalog Volumes; no Google Drive mount is required.

import os
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "")

from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))

In [ ]:
import os, re, ast
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# ---- 路径（按你刚才描述）----
IN_UNIFIED_14 = Path("/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.1.4.parquet")
PATH_M1       = Path("/Volumes/main/default/thesis_project/M1/Test_2.2/qasper_test_M1_answers_sample_2.2.parquet")
PATH_M3       = Path("/Volumes/main/default/thesis_project/M3/QASPER_M3_Test/Q_pred_test_M3_1.4.parquet")
OUT_UNIFIED_16= Path("/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.1.5.parquet")
CHUNK_INDEX_PATH = None

In [ ]:
# ================================== 小工具 ==================================
def coerce_list(x):
    if isinstance(x, list):  return [str(t) for t in x]
    if isinstance(x, tuple): return [str(t) for t in x]
    if isinstance(x, str):
        s = x.strip()
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            try:
                v = ast.literal_eval(s)
                if isinstance(v, (list, tuple)): return [str(t) for t in v]
            except Exception:
                pass
        return [s] if s else []
    return []

def detect_chunk_cols(df_cols):
    """
    在切片库里找 (id列, 文本列)。常见组合：
    ('chunk_id' or 'id' or 'pid' or 'chunk') + ('text' or 'content' or 'passage' or 'chunk_text')
    """
    id_candidates   = ["chunk_id","id","pid","chunk","doc_id","passage_id"]
    text_candidates = ["text","content","passage","chunk_text","body","segment","paragraph"]
    id_col = next((c for c in id_candidates   if c in df_cols), None)
    tx_col = next((c for c in text_candidates if c in df_cols), None)
    return id_col, tx_col

def build_chunk_map_from_parquet(p: Path, max_rows=2_000_000):
    df = pd.read_parquet(p)
    id_col, tx_col = detect_chunk_cols(df.columns)
    if not id_col or not tx_col:
        raise ValueError(f"切片库 {p} 里找不到 id/text 列（现有列：{list(df.columns)}）")
    # 只保留两列，加快映射
    slim = df[[id_col, tx_col]].dropna().drop_duplicates(subset=[id_col])
    mp = dict(zip(slim[id_col].astype(str), slim[tx_col].astype(str)))
    return mp, id_col, tx_col, len(slim)

def auto_find_chunk_index(root="/Volumes/main/default/thesis_project", max_hits=5):
    """
    粗略扫描 Drive 下的 parquet/jsonl，看有没有包含 chunk id + 文本的文件。
    命中后返回第一个候选路径列表（你也可以挑别的）。
    """
    rootp = Path(root)
    candidates = []
    for ext in ("*.parquet","*.jsonl","*.json"):
        for p in rootp.rglob(ext):
            name = p.name.lower()
            if any(kw in name for kw in ["chunk","passage","corpus","index","qasper","retriev","doc"]):
                candidates.append(p)
                if len(candidates) >= max_hits:
                    return candidates
    return candidates

In [ ]:
# ================================== 1) 读取三个输入 ==================================
DFu = pd.read_parquet(IN_UNIFIED_14)
M1  = pd.read_parquet(PATH_M1)
M3  = pd.read_parquet(PATH_M3)

print(f"Unified 1.4 rows={len(DFu)}  | cols={len(DFu.columns)}")
print(f"M1 rows={len(M1)} cols={len(M1.columns)} | M3 rows={len(M3)} cols={len(M3.columns)}")

In [ ]:
# 规范列
if "retrieved_ctx" not in DFu.columns:
    DFu["retrieved_ctx"] = [[] for _ in range(len(DFu))]
DFu["retrieved_ctx"] = DFu["retrieved_ctx"].apply(coerce_list)

# M1：retrieved_chunks 是 ID 列
assert "question" in M1.columns and "retrieved_chunks" in M1.columns, "M1 缺少 question 或 retrieved_chunks"
M1["retrieved_chunks"] = M1["retrieved_chunks"].apply(coerce_list)

# M3：contexts 已是文本片段
if "contexts" in M3.columns:
    M3["contexts"] = M3["contexts"].apply(coerce_list)
else:
    M3["contexts"] = [[] for _ in range(len(M3))]

In [ ]:
# ================================== 2) 准备切片库映射 ==================================
chunk_map = {}
if CHUNK_INDEX_PATH is not None:
    CHUNK_INDEX_PATH = Path(CHUNK_INDEX_PATH)
    print("使用指定切片库：", CHUNK_INDEX_PATH)
    chunk_map, id_col, tx_col, nrows = build_chunk_map_from_parquet(CHUNK_INDEX_PATH)
    print(f"已加载切片库：{CHUNK_INDEX_PATH.name}  行={nrows:,}  id列={id_col}  文本列={tx_col}")
else:
    # 自动扫描可能的切片库
    hits = auto_find_chunk_index()
    if hits:
        print("🔎 发现以下可能的切片库候选（按相关性随便取一个，或把路径填到 CHUNK_INDEX_PATH）：")
        for p in hits:
            print("  -", p)
        # 先尝试第一个候选（有可能失败，失败请把确切路径填到 CHUNK_INDEX_PATH 再跑一次）
        try:
            chunk_map, id_col, tx_col, nrows = build_chunk_map_from_parquet(hits[0])
            print(f"✅ 试用候选切片库：{hits[0].name}  行={nrows:,}  id列={id_col}  文本列={tx_col}")
        except Exception as e:
            print("⚠️ 自动加载候选失败：", e)
            print("👉 请把真实切片库 parquet 路径填到 CHUNK_INDEX_PATH 后重跑本单元。")
    else:
        print("⚠️ 未在 Drive 下自动发现切片库。请设置 CHUNK_INDEX_PATH 为你的语料切片 parquet。")

In [ ]:
import pandas as pd, numpy as np, re, json, ast
from pathlib import Path

# ---- 路径 ----
IN_UNIFIED = Path("/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.1.4.parquet")
PATH_M1    = Path("/Volumes/main/default/thesis_project/M1/Test_2.2/qasper_test_M1_answers_sample_2.2.parquet")
PATH_M3    = Path("/Volumes/main/default/thesis_project/M3/QASPER_M3_Test/Q_pred_test_M3_1.4.parquet")
META_JSONL = Path("/Volumes/main/default/thesis_project/M1/Test_2.2/qasper_test_meta.jsonl")

OUT_UNIFIED = Path("/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.1.7.parquet")


In [ ]:
# ---- 工具：健壮解析 retrieved_chunks（支持无引号列表字符串）----
def parse_chunk_id_list(x):
    if isinstance(x, list):
        return [str(t).strip().strip("'").strip('"') for t in x if str(t).strip()]
    if isinstance(x, tuple):
        return [str(t).strip().strip("'").strip('"') for t in x if str(t).strip()]
    if isinstance(x, str):
        s = x.strip()
        # 去掉首尾方括号
        if s.startswith("[") and s.endswith("]"):
            inner = s[1:-1]
        else:
            inner = s
        # 先尝试按逗号切
        parts = [p.strip().strip("'").strip('"') for p in inner.split(",") if p.strip()]
        # 如果只有一个元素且里面仍包含逗号/空格，再用正则兜底
        if len(parts) == 1 and ("::" in parts[0] and "," in parts[0]):
            parts = re.findall(r"[^\s,\[\]]+", inner)
        return [p for p in parts if p]
    return []

def normalize_question(q: str) -> str:
    return re.sub(r"\s+", " ", str(q or "").strip().lower())

def first_nonempty_list(series: pd.Series):
    for v in series:
        if isinstance(v, (list, tuple)) and len(v) > 0:
            return list(v)
    return []

In [ ]:
# ---- 读取数据 ----
DFu = pd.read_parquet(IN_UNIFIED)
M1  = pd.read_parquet(PATH_M1)
M3  = pd.read_parquet(PATH_M3)
print(f"Loaded unified 1.4: rows={len(DFu)}, cols={len(DFu.columns)}")
print(f"M1 rows={len(M1)}, M3 rows={len(M3)}")

# 标准化 key（按 question 文本）
DFu["__qkey__"] = DFu["question"].apply(normalize_question)
M1["__qkey__"]  = M1["question"].apply(normalize_question)
M3["__qkey__"]  = M3["question"].apply(normalize_question)

In [ ]:
# ---- 载入 meta.jsonl，建立双键映射 ----
meta_df = pd.read_json(META_JSONL, lines=True)
need = ["paper_id","chunk_id","text"]
assert all(c in meta_df.columns for c in need), f"meta JSONL 缺列，现有：{list(meta_df.columns)}"

meta_df = meta_df.dropna(subset=["chunk_id","text"]).drop_duplicates(subset=["paper_id","chunk_id"], keep="first")
meta_df["paper_id"] = meta_df["paper_id"].astype(str)
meta_df["chunk_id"] = meta_df["chunk_id"].astype(str)
meta_df["__combo__"] = meta_df["paper_id"] + "::" + meta_df["chunk_id"]

map_combo = dict(zip(meta_df["__combo__"], meta_df["text"].astype(str)))
map_id    = dict(zip(meta_df["chunk_id"], meta_df["text"].astype(str)))

print(f"chunk map sizes -> combo: {len(map_combo):,} | id-only: {len(map_id):,}")

In [ ]:
# ---- 解析 M1 的 retrieved_chunks 并尝试映射 ----
M1["retrieved_chunks_parsed"] = M1["retrieved_chunks"].apply(parse_chunk_id_list)

In [ ]:
# 日志：看几条解析结果
print("M1 retrieved_chunks 示例（解析后前3条）:")
for i in range(min(3, len(M1))):
    print("  ", M1.loc[i, "retrieved_chunks_parsed"][:5])

In [ ]:
# ---- 工具：打印帮助 ----
def _prepr(x, w=160):
    s = repr(x)
    return s if len(s) <= w else s[:w] + "…"

def normalize_question(q: str) -> str:
    return re.sub(r"\s+", " ", str(q or "").strip().lower())

In [ ]:
# ---- 1) 读取 ----
DFu = pd.read_parquet(IN_UNIFIED)
M1  = pd.read_parquet(PATH_M1)
M3  = pd.read_parquet(PATH_M3)
print(f"Unified rows={len(DFu)} | M1 rows={len(M1)} | M3 rows={len(M3)}")

assert "retrieved_chunks" in M1.columns, "M1 缺少列 retrieved_chunks"
print("\n[诊断] M1.retrieved_chunks 前 12 条（原始 repr）:")
for i, v in enumerate(M1["retrieved_chunks"].head(12).tolist()):
    print(f"  [{i}] type={type(v).__name__} | {_prepr(v)}")

In [ ]:
# ---- 2) 强健解析器：支持多种形态（list / tuple / dict-list / JSON 字符串 / 方括号字符串 / 空白分隔 / 逗号分隔）----
def parse_chunk_ids(x):
    # 已经是 list/tuple
    if isinstance(x, (list, tuple)):
        out = []
        for t in x:
            if isinstance(t, dict):
                # 常见键位
                for k in ("chunk_id","chunkId","id","ctx_id"):
                    if k in t and str(t[k]).strip():
                        out.append(str(t[k]).strip())
                        break
            elif isinstance(t, (str, int, float)):
                s = str(t).strip().strip("'").strip('"')
                if s: out.append(s)
        return out

    # None / NaN
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []

    s = str(x).strip()

    # JSON 列表（带引号或 dict）
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("{") and s.endswith("}")):
        try:
            obj = json.loads(s)
            return parse_chunk_ids(obj)
        except Exception:
            # 非严格 JSON（比如无引号 token 的 [a, b]），继续走下方解析
            pass

    # 去掉外层方括号（如有）
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1].strip()

    # 优先按逗号分
    if "," in s:
        parts = [p.strip().strip("'").strip('"') for p in s.split(",")]
        parts = [p for p in parts if p]
    else:
        # 再按空白分
        parts = [p.strip().strip("'").strip('"') for p in re.split(r"\s+", s) if p.strip()]

    # 最后兜底：提取“非空白且不含逗号/中括号”的片段
    if not parts:
        parts = re.findall(r"[^\s,\[\]]+", s)

    # 过滤一下明显无关的符号
    parts = [p for p in parts if p not in ("[","]")]
    return parts

In [ ]:
M1["retrieved_chunks_parsed"] = M1["retrieved_chunks"].apply(parse_chunk_ids)

print("\n[诊断] M1.retrieved_chunks_parsed 前 12 条（解析后）:")
for i, v in enumerate(M1["retrieved_chunks_parsed"].head(12).tolist()):
    print(f"  [{i}] {v[:6]}")

n_nonempty = int(M1["retrieved_chunks_parsed"].apply(lambda a: isinstance(a, list) and len(a)>0).sum())
print(f"解析后非空行数：{n_nonempty} / {len(M1)}")

In [ ]:
# ===== 将 M1 的 chunk_id → 文本（用 meta.jsonl），合并 M3 的 contexts，写出 1.8 版统一表 =====
import pandas as pd, numpy as np, re, json, ast
from pathlib import Path

# ---- 路径 ----
IN_UNIFIED   = Path("/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.1.4.parquet")
PATH_M1      = Path("/Volumes/main/default/thesis_project/M1/Test_2.2/qasper_test_M1_answers_sample_2.2.parquet")
PATH_M3      = Path("/Volumes/main/default/thesis_project/M3/QASPER_M3_Test/Q_pred_test_M3_1.4.parquet")
META_JSONL   = Path("/Volumes/main/default/thesis_project/M1/Test_2.2/qasper_test_meta.jsonl")
OUT_UNIFIED  = Path("/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.1.5.parquet")

# ---- 工具 ----
def normalize_question(q: str) -> str:
    return re.sub(r"\s+", " ", str(q or "").strip().lower())

def coerce_list(x):
    if isinstance(x, list):  return [str(t) for t in x]
    if isinstance(x, tuple): return [str(t) for t in x]
    if isinstance(x, str):
        s = x.strip()
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            try:
                v = ast.literal_eval(s)
                if isinstance(v, (list, tuple)): return [str(t) for t in v]
            except Exception:
                pass
        return [s] if s else []
    return []

def first_nonempty_list(series: pd.Series):
    for v in series:
        if isinstance(v, (list, tuple)) and len(v) > 0:
            return list(v)
    return []

# ---- 读取统一文件 + M1 + M3 ----
DFu = pd.read_parquet(IN_UNIFIED)
M1  = pd.read_parquet(PATH_M1)
M3  = pd.read_parquet(PATH_M3)

# 规范 key（按 question）
DFu["__qkey__"] = DFu["question"].apply(normalize_question)
M1["__qkey__"]  = M1["question"].apply(normalize_question)
M3["__qkey__"]  = M3["question"].apply(normalize_question)

# ---- 1) 读 meta.jsonl，建立双键映射（paper::chunk 与裸 chunk）----
meta_df = pd.read_json(META_JSONL, lines=True)
need = ["paper_id","chunk_id","text"]
assert all(c in meta_df.columns for c in need), f"meta JSONL 缺列：{list(meta_df.columns)}"
meta_df = meta_df.dropna(subset=["chunk_id","text"]).drop_duplicates(subset=["paper_id","chunk_id"], keep="first")
meta_df["paper_id"] = meta_df["paper_id"].astype(str)
meta_df["chunk_id"] = meta_df["chunk_id"].astype(str)
meta_df["__combo__"] = meta_df["paper_id"] + "::" + meta_df["chunk_id"]
map_combo = dict(zip(meta_df["__combo__"], meta_df["text"].astype(str)))
map_id    = dict(zip(meta_df["chunk_id"], meta_df["text"].astype(str)))
print(f"chunk map sizes -> combo: {len(map_combo):,} | id-only: {len(map_id):,}")

# ---- 2) 解析 M1 的 retrieved_chunks（如果已有 parsed 列就复用）----
def parse_chunk_ids(x):
    if isinstance(x, (list, tuple)):
        out=[]
        for t in x:
            if isinstance(t, dict):
                for k in ("chunk_id","chunkId","id","ctx_id"):
                    if k in t and str(t[k]).strip():
                        out.append(str(t[k]).strip()); break
            else:
                s = str(t).strip().strip("'").strip('"')
                if s: out.append(s)
        return out
    if x is None or (isinstance(x, float) and np.isnan(x)): return []
    s = str(x).strip()
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("{") and s.endswith("}")):
        try:
            obj = json.loads(s); return parse_chunk_ids(obj)
        except Exception: pass
    if s.startswith("[") and s.endswith("]"): s = s[1:-1].strip()
    parts = [p.strip().strip("'").strip('"') for p in s.split(",")] if "," in s else \
            [p.strip().strip("'").strip('"') for p in re.split(r"\s+", s) if p.strip()]
    if not parts: parts = re.findall(r"[^\s,\[\]]+", s)
    parts = [p for p in parts if p not in ("[","]")]
    return parts

if "retrieved_chunks_parsed" not in M1.columns:
    M1["retrieved_chunks_parsed"] = M1["retrieved_chunks"].apply(parse_chunk_ids)

# ---- 3) ID → 文本映射（优先 combo，其次右半段，再次裸 chunk 或 combo by paper_id）----
def ids_to_texts(ids: list, paper_id: str = None):
    if not isinstance(ids, list) or not ids: return [], []
    out, miss = [], []
    for tok in ids:
        t = None
        tok = str(tok)
        if "::" in tok:
            t = map_combo.get(tok)
            if t is None:
                right = tok.split("::",1)[1]
                t = map_id.get(right)
        else:
            if paper_id and (paper_id + "::" + tok) in map_combo:
                t = map_combo[paper_id + "::" + tok]
            else:
                t = map_id.get(tok)
        if t: out.append(t)
        else: miss.append(tok)
    return out, miss

pid_col = "paper_id" if "paper_id" in M1.columns else None
tm = M1.apply(lambda r: ids_to_texts(r["retrieved_chunks_parsed"], str(r[pid_col]) if pid_col and pd.notna(r[pid_col]) else None), axis=1)
M1["retrieved_ctx_text"] = [x[0] for x in tm]
M1["__miss_tokens__"]    = [x[1] for x in tm]

# 覆盖率统计
total_ids   = int(sum(len(v) for v in M1["retrieved_chunks_parsed"]))
matched_ids = int(sum(len(v) for v in M1["retrieved_ctx_text"]))
rows_nonempty = int(M1["retrieved_ctx_text"].apply(lambda x: isinstance(x,list) and len(x)>0).sum())
print(f"mapped rows with non-empty ctx: {rows_nonempty}/{len(M1)} | matched ids: {matched_ids}/{total_ids}")

miss_flat = [z for lst in M1["__miss_tokens__"] if isinstance(lst, list) for z in lst]
print("sample missed tokens (<=10):", miss_flat[:10])

# ---- 4) 汇总成问题级映射并合入统一表（M1 用映射文本，M3 用 contexts 文本）----
M1_map = M1.groupby("__qkey__")["retrieved_ctx_text"].apply(first_nonempty_list)

M3["contexts"] = M3["contexts"].apply(coerce_list)
M3_map = M3.groupby("__qkey__")["contexts"].apply(first_nonempty_list)

if "retrieved_ctx" not in DFu.columns:
    DFu["retrieved_ctx"] = [[] for _ in range(len(DFu))]
else:
    DFu["retrieved_ctx"] = DFu["retrieved_ctx"].apply(lambda v: v if isinstance(v, list) else [])

is_m1 = DFu["model"].astype(str).str.upper().eq("M1")
is_m3 = DFu["model"].astype(str).str.upper().eq("M3")

DF_out = DFu.copy()
DF_out.loc[is_m1, "retrieved_ctx"] = DF_out.loc[is_m1, "__qkey__"].map(M1_map).apply(lambda v: v if isinstance(v, list) else [])
DF_out.loc[is_m3, "retrieved_ctx"] = DF_out.loc[is_m3, "__qkey__"].map(M3_map).apply(lambda v: v if isinstance(v, list) else [])
DF_out["has_retrieval"] = DF_out["retrieved_ctx"].apply(lambda x: isinstance(x, list) and len(x)>0)
DF_out.drop(columns=["__qkey__"], inplace=True)

# ---- 5) 写出与检查 ----
OUT_UNIFIED.parent.mkdir(parents=True, exist_ok=True)
DF_out.to_parquet(OUT_UNIFIED, index=False)
print("💾 wrote:", OUT_UNIFIED)

print("\nNon-empty retrieved_ctx by model:")
print(DF_out.assign(nonempty=DF_out["retrieved_ctx"].apply(lambda x: int(isinstance(x,list) and len(x)>0))) \
      .groupby("model")["nonempty"].sum())

tmp = DF_out.copy()
tmp["retr_len"] = tmp["retrieved_ctx"].apply(lambda v: len(v) if isinstance(v,list) else 0)
print("\nSample rows (model, question_id, retr_len):")
print(tmp[["model","question_id","question","retr_len"]].head(10).to_string(index=False))


In [ ]:
# ===== Robust mapping: M1 chunk_id → text (meta.jsonl with variants) -> write unified 1.9 =====
import pandas as pd, numpy as np, re, json, ast
from pathlib import Path
from collections import defaultdict

# ---- paths ----
IN_UNIFIED   = Path("/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.1.4.parquet")
PATH_M1      = Path("/Volumes/main/default/thesis_project/M1/Test_2.2/qasper_test_M1_answers_sample_2.2.parquet")
PATH_M3      = Path("/Volumes/main/default/thesis_project/M3/QASPER_M3_Test/Q_pred_test_M3_1.4.parquet")
META_JSONL   = Path("/Volumes/main/default/thesis_project/M1/Test_2.2/qasper_test_meta.jsonl")

OUT_UNIFIED  = Path("/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.1.6.parquet")

# ---- helpers ----
def norm_q(s): return re.sub(r"\s+", " ", str(s or "").strip().lower())

def parse_chunk_ids(x):
    if isinstance(x, (list, tuple)):
        out=[]
        for t in x:
            if isinstance(t, dict):
                for k in ("chunk_id","chunkId","id","ctx_id"):
                    if k in t and str(t[k]).strip():
                        out.append(str(t[k]).strip()); break
            else:
                s = str(t).strip().strip("'").strip('"')
                if s: out.append(s)
        return out
    if x is None or (isinstance(x, float) and np.isnan(x)): return []
    s = str(x).strip()
    # JSON?
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("{") and s.endswith("}")):
        try:
            obj = json.loads(s); return parse_chunk_ids(obj)
        except Exception: pass
    # strip brackets
    if s.startswith("[") and s.endswith("]"): s = s[1:-1]
    parts = [p.strip().strip("'").strip('"') for p in s.split(",")] if "," in s else \
            [p.strip().strip("'").strip('"') for p in re.split(r"\s+", s) if p.strip()]
    if not parts:
        parts = re.findall(r"[^\s,\[\]]+", s)
    return [p for p in parts if p not in ("[","]")]

def pid_variants(pid):
    """paper_id 变体：原样、去前缀、加前缀、去版本、加 v1..v9"""
    if pid is None: return set()
    s = str(pid)
    out = set([s])
    # 去/加 arXiv: 前缀
    base = s
    if base.lower().startswith("arxiv:"): base = base.split(":",1)[1]
    out.add(base)
    out.add("arXiv:" + base)
    # 去版本
    nov = re.sub(r"v\d+$","", base)
    out.add(nov)
    out.add("arXiv:" + nov)
    # 常见版本补充
    for v in range(1,10):
        out.add(f"{nov}v{v}")
        out.add(f"arXiv:{nov}v{v}")
    return out

def cid_variants(cid):
    """chunk_id 变体：原样、去下划线、不同 #c0 组合、sXpY模式归一化等"""
    if cid is None: return set()
    s = str(cid)
    out = set([s])
    s_no_us = s.replace("_","")
    out.add(s_no_us)

    # 如果有 '#cN'，加入去掉 '#cN' 的版本
    m = re.match(r"^(.*?)(#c\d+)$", s_no_us)
    if m:
        out.add(m.group(1))  # 去掉 #cN
    # 把 'sX_pY_cZ' / 'sXpYcZ' / 'sX.pY.cZ' 等统一为 'sXpY#cZ'
    rx = re.match(r"^s(\d+)[\._-]*p(\d+)(?:[\._-]*c(\d+))?$", s_no_us)
    if rx:
        S,P,C = rx.group(1), rx.group(2), rx.group(3) or "0"
        out.update({
            f"s{S}p{P}#c{C}",
            f"s{S}p{P}",
            f"s{S}p{P}c{C}",
            f"s{S}_p{P}_c{C}",
            f"s{S}_p{P}",
            f"s{S}.p{P}.c{C}",
        })
    # abstract / conclusion 等别名试探
    if s_no_us.lower().startswith("abstract"):
        out.update(["abstract","abstract#c0","abs","abs#c0"])
    if s_no_us.lower().startswith("conclusion"):
        out.update(["conclusion","conclusion#c0"])

    return out

# ---- 1) read data ----
DFu = pd.read_parquet(IN_UNIFIED)
M1  = pd.read_parquet(PATH_M1)
M3  = pd.read_parquet(PATH_M3)

DFu["__qkey__"] = DFu["question"].apply(norm_q)
M1["__qkey__"]  = M1["question"].apply(norm_q)
M3["__qkey__"]  = M3["question"].apply(norm_q)

# ---- 2) read meta and build variant maps ----
meta = pd.read_json(META_JSONL, lines=True)
assert all(c in meta.columns for c in ("paper_id","chunk_id","text")), f"meta.jsonl 缺列: {list(meta.columns)}"

meta = meta.dropna(subset=["paper_id","chunk_id","text"]).drop_duplicates(subset=["paper_id","chunk_id"], keep="first")
meta["paper_id"] = meta["paper_id"].astype(str)
meta["chunk_id"] = meta["chunk_id"].astype(str)

# 建立两类映射：裸 chunk / 组合键
map_id    = {}
map_combo = {}

for pid, cid, txt in meta[["paper_id","chunk_id","text"]].itertuples(index=False):
    pvars = pid_variants(pid)
    cvars = cid_variants(cid)
    # 裸 chunk（保守：只记录第一次出现，避免覆盖）
    for cv in cvars:
        if cv not in map_id:
            map_id[cv] = txt
    # 组合键
    for pv in pvars:
        for cv in cvars:
            key = f"{pv}::{cv}"
            if key not in map_combo:
                map_combo[key] = txt

print(f"variant maps built -> id-only: {len(map_id):,} | combo: {len(map_combo):,}")

# ---- 3) parse M1 retrieved_chunks and map ----
M1["retrieved_chunks_parsed"] = M1["retrieved_chunks"].apply(parse_chunk_ids)
pid_col = "paper_id" if "paper_id" in M1.columns else None

def map_ids(ids, pid):
    out, miss = [], []
    pvars = pid_variants(pid) if pid is not None else set()
    for tok in ids:
        t = None
        # 1) token 自带 paper::cid
        if "::" in tok:
            # 原样
            t = map_combo.get(tok)
            # 右半段
            if t is None:
                right = tok.split("::",1)[1]
                t = map_id.get(right)
            # 再试右半段的变体
            if t is None:
                for cv in cid_variants(right):
                    t = map_id.get(cv)
                    if t: break
            # 再试 token 的变体
            if t is None:
                for pv in pid_variants(tok.split("::",1)[0]):
                    for cv in cid_variants(right):
                        t = map_combo.get(f"{pv}::{cv}")
                        if t: break
                    if t: break
        else:
            # 2) 无 paper 的 token：先裸 chunk，再 pvar::cvar
            t = map_id.get(tok)
            if t is None:
                for cv in cid_variants(tok):
                    t = map_id.get(cv)
                    if t: break
            if t is None and pvars:
                found=False
                for pv in pvars:
                    for cv in cid_variants(tok):
                        t = map_combo.get(f"{pv}::{cv}")
                        if t:
                            found=True; break
                    if found: break
        if t: out.append(t)
        else: miss.append(tok)
    return out, miss

mm = M1.apply(lambda r: map_ids(r["retrieved_chunks_parsed"], r.get(pid_col)) , axis=1)
M1["retrieved_ctx_text"] = [x[0] for x in mm]
M1["__miss_tokens__"]    = [x[1] for x in mm]

total_ids   = int(sum(len(v) for v in M1["retrieved_chunks_parsed"]))
matched_ids = int(sum(len(v) for v in M1["retrieved_ctx_text"]))
rows_nonempty = int(M1["retrieved_ctx_text"].apply(lambda x: isinstance(x,list) and len(x)>0).sum())
print(f"mapped rows with non-empty ctx: {rows_nonempty}/{len(M1)} | matched ids: {matched_ids}/{total_ids}")

miss_flat = [z for lst in M1["__miss_tokens__"] if isinstance(lst, list) for z in lst]
print("sample missed tokens (<=15):", miss_flat[:15])

# ---- 4) merge back to unified (M1 uses mapped text; M3 uses its own contexts) ----
def first_nonempty_list(series: pd.Series):
    for v in series:
        if isinstance(v, (list, tuple)) and len(v) > 0:
            return list(v)
    return []

M1_map = M1.groupby("__qkey__")["retrieved_ctx_text"].apply(first_nonempty_list)

M3["contexts"] = M3["contexts"].apply(lambda v: v if isinstance(v, list) else parse_chunk_ids(v))
M3_map = M3.groupby("__qkey__")["contexts"].apply(first_nonempty_list)

DFu["retrieved_ctx"] = DFu.get("retrieved_ctx", [[]]*len(DFu))
DFu["retrieved_ctx"] = DFu["retrieved_ctx"].apply(lambda v: v if isinstance(v, list) else [])

is_m1 = DFu["model"].astype(str).str.upper().eq("M1")
is_m3 = DFu["model"].astype(str).str.upper().eq("M3")

DF_out = DFu.copy()
DF_out.loc[is_m1, "retrieved_ctx"] = DF_out.loc[is_m1, "__qkey__"].map(M1_map).apply(lambda v: v if isinstance(v, list) else [])
DF_out.loc[is_m3, "retrieved_ctx"] = DF_out.loc[is_m3, "__qkey__"].map(M3_map).apply(lambda v: v if isinstance(v, list) else [])
DF_out["has_retrieval"] = DF_out["retrieved_ctx"].apply(lambda x: isinstance(x, list) and len(x)>0)
DF_out.drop(columns=["__qkey__"], inplace=True)

OUT_UNIFIED.parent.mkdir(parents=True, exist_ok=True)
DF_out.to_parquet(OUT_UNIFIED, index=False)
print("💾 wrote:", OUT_UNIFIED)

print("\nNon-empty retrieved_ctx by model:")
print(DF_out.assign(nonempty=DF_out["retrieved_ctx"].apply(lambda x: int(isinstance(x,list) and len(x)>0))) \
      .groupby("model")["nonempty"].sum())


In [ ]:
# ============= 诊断 + 增强映射（paper_id 变体：点/下划线/去/加版本/加前缀；chunk_id 变体） =============
import pandas as pd, numpy as np, re, json, ast
from pathlib import Path
from collections import defaultdict, Counter

# ---- 输入路径 ----
PATH_M1    = Path("/Volumes/main/default/thesis_project/M1/Test_2.2/qasper_test_M1_answers_sample_2.2.parquet")
META_JSONL = Path("/Volumes/main/default/thesis_project/M1/Test_2.2/qasper_test_meta.jsonl")

# ---- 读取 ----
M1 = pd.read_parquet(PATH_M1)
meta = pd.read_json(META_JSONL, lines=True)
assert all(c in meta.columns for c in ("paper_id","chunk_id","text")), f"meta.jsonl 缺列: {list(meta.columns)}"

# ---- 解析 M1 的 retrieved_chunks ----
def parse_chunk_ids(x):
    if isinstance(x, (list, tuple)):
        out=[]
        for t in x:
            if isinstance(t, dict):
                for k in ("chunk_id","chunkId","id","ctx_id"):
                    if k in t and str(t[k]).strip():
                        out.append(str(t[k]).strip()); break
            else:
                s = str(t).strip().strip("'").strip('"')
                if s: out.append(s)
        return out
    if x is None or (isinstance(x, float) and np.isnan(x)): return []
    s = str(x).strip()
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("{") and s.endswith("}")):
        try:
            obj = json.loads(s); return parse_chunk_ids(obj)
        except Exception: pass
    if s.startswith("[") and s.endswith("]"): s = s[1:-1]
    parts = [p.strip().strip("'").strip('"') for p in s.split(",")] if "," in s else \
            [p.strip().strip("'").strip('"') for p in re.split(r"\s+", s) if p.strip()]
    if not parts: parts = re.findall(r"[^\s,\[\]]+", s)
    return [p for p in parts if p not in ("[","]")]

M1["retrieved_chunks_parsed"] = M1["retrieved_chunks"].apply(parse_chunk_ids)

# ---- 从未命中的样例提取若干 paper_id（左半段）用于诊断 ----
# 先随便取 20 个 token
tokens = []
for lst in M1["retrieved_chunks_parsed"].head(50):
    for t in lst:
        if "::" in t:
            tokens.append(t)
    if len(tokens) >= 20:
        break

miss_pids = []
for tok in tokens[:20]:
    pid = tok.split("::",1)[0]
    miss_pids.append(pid)
miss_pids = list(dict.fromkeys(miss_pids))  # 去重保序

print("🔎 将诊断这些 paper_id 的真实形态（来自 meta）：", miss_pids[:5])

# ---- 构建 meta 的基准字段（便于对齐）----
meta = meta.dropna(subset=["paper_id","chunk_id","text"]).drop_duplicates(subset=["paper_id","chunk_id"], keep="first")
meta["paper_id"] = meta["paper_id"].astype(str)
meta["chunk_id"] = meta["chunk_id"].astype(str)

# 定义更强的 paper_id 变体：点<->下划线、去/加 arXiv、去版本、加 v1..v9
def pid_variants(pid):
    s = str(pid)
    out = set()
    # 原样 + arXiv 前缀变体
    variants = {s}
    if s.lower().startswith("arxiv:"):
        variants.add(s.split(":",1)[1])
    else:
        variants.add("arXiv:" + s)
    # 点/下划线互换、去点
    tmp = set()
    for v in variants:
        tmp.add(v)
        tmp.add(v.replace(".", "_"))
        tmp.add(v.replace(".", ""))  # 去掉点
    variants = tmp
    # 去/保版本
    base_set = set()
    for v in variants:
        base_set.add(re.sub(r"v\d+$","", v))
    # 为每个 base，生成 v1..v9 版本
    allv = set()
    for b in base_set:
        allv.add(b)
        for i in range(1,10):
            allv.add(f"{b}v{i}")
    # 再为每个生成带/不带 arXiv: 的版本
    final = set()
    for a in allv:
        final.add(a)
        if not a.lower().startswith("arxiv:"):
            final.add("arXiv:" + a)
    return final

# chunk_id 变体：原样、去下划线、去 #cN、cN 多种写法、sXpY 标准化
def cid_variants(cid):
    s = str(cid)
    out = {s, s.replace("_","")}
    s0 = s.replace("_","")
    m = re.match(r"^(.*?)(#c\d+)$", s0)
    if m:
        out.add(m.group(1))                     # 去 #cN
        out.add(m.group(1)+"c"+m.group(2)[2:])  # sXpYcN
        out.add(m.group(1)+"_c"+m.group(2)[2:]) # sXpY_cN
    # sXpY 标准化（容错 . _ -）
    rx = re.match(r"^s(\d+)[\._-]*p(\d+)(?:[\._-]*c(\d+))?$", s0)
    if rx:
        S,P,C = rx.group(1), rx.group(2), rx.group(3) or "0"
        out |= {
            f"s{S}p{P}#c{C}", f"s{S}p{P}", f"s{S}p{P}c{C}",
            f"s{S}_p{P}_c{C}", f"s{S}_p{P}", f"s{S}.p{P}.c{C}",
        }
    if s0.lower().startswith("abstract"):
        out |= {"abstract","abstract#c0","abs","abs#c0"}
    if s0.lower().startswith("conclusion"):
        out |= {"conclusion","conclusion#c0"}
    return out

# ---- 先“看一眼”：meta 里对应 paper 的真实样子 ----
for pid in miss_pids[:3]:
    pvars = pid_variants(pid)
    sub = meta[meta["paper_id"].isin(pvars)]
    print(f"\n=== 诊断 paper_id {pid} 命中的 meta 行数: {len(sub)} ===")
    if len(sub):
        # 展示这个 paper 的 chunk_id 取样
        samp = sub["chunk_id"].head(15).tolist()
        print("chunk_id 示例：", samp)
    else:
        # 尝试松匹配：去点、下划线互换后包含关系
        b = pid.replace(".","")
        sub2 = meta[meta["paper_id"].str.replace(".","", regex=False).str.contains(b)]
        print("（严格变体未命中，松匹配包含关系命中行数）:", len(sub2))
        if len(sub2):
            print("paper_id 示例：", sub2["paper_id"].head(5).tolist())
            print("chunk_id 示例：", sub2["chunk_id"].head(10).tolist())

# ---- 建立大映射：裸 chunk & 组合（全部变体）----
map_id    = {}
map_combo = {}
for _, row in meta.iterrows():
    pvars = pid_variants(row["paper_id"])
    cvars = cid_variants(row["chunk_id"])
    for cv in cvars:
        map_id.setdefault(cv, row["text"])
    for pv in pvars:
        for cv in cvars:
            map_combo.setdefault(f"{pv}::{cv}", row["text"])
print(f"\nvariant maps built -> id-only: {len(map_id):,} | combo: {len(map_combo):,}")

# ---- 测试 100 个 token 的命中率 ----
test_tokens = []
for lst in M1["retrieved_chunks_parsed"]:
    for t in lst:
        test_tokens.append(t)
    if len(test_tokens) >= 100:
        break

def map_one_token(tok):
    if "::" in tok:
        if tok in map_combo: return True
        left, right = tok.split("::",1)
        # 尝试通过变体组合
        for pv in pid_variants(left):
            for cv in cid_variants(right):
                if f"{pv}::{cv}" in map_combo: return True
        # 尝试右半段在裸 id 里
        for cv in cid_variants(right):
            if cv in map_id: return True
        return False
    else:
        if tok in map_id: return True
        for cv in cid_variants(tok):
            if cv in map_id: return True
        return False

hit = sum(map_one_token(t) for t in test_tokens)
print(f"小样本命中：{hit}/{len(test_tokens)}")

# ---- 如果命中率显著>0，再对全量做映射（只打印分数，不写回；确认后你再跑全量写回那格） ----
if hit == 0:
    print("\n❗ 小样本仍 0。请看上面的“诊断输出”：")
    print("  - meta 中对应 paper 的 paper_id/ chunk_id 样例是什么样？")
    print("  - 如果和 token 的形态仍然有差异，把那几行样例发我（或告诉我具体差异），我再补规则。")
else:
    print("\n✅ 规则有效：可以用这套变体构造进行全量映射并回填统一表。")

In [ ]:
# ===== 修复版：用 “完整 chunk_id” 直接映射（主），右半段作为备选 → 回填到统一表 =====
import pandas as pd, numpy as np, re, json, ast
from pathlib import Path

# 路径
IN_UNIFIED   = Path("/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.1.4.parquet")
PATH_M1      = Path("/Volumes/main/default/thesis_project/M1/Test_2.2/qasper_test_M1_answers_sample_2.2.parquet")
PATH_M3      = Path("/Volumes/main/default/thesis_project/M3/QASPER_M3_Test/Q_pred_test_M3_1.4.parquet")
META_JSONL   = Path("/Volumes/main/default/thesis_project/M1/Test_2.2/qasper_test_meta.jsonl")
OUT_UNIFIED  = Path("/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.1.7.parquet")

def norm_q(s): return re.sub(r"\s+", " ", str(s or "").strip().lower())

def parse_chunk_ids(x):
    if isinstance(x, (list, tuple)):
        out=[]
        for t in x:
            if isinstance(t, dict):
                for k in ("chunk_id","chunkId","id","ctx_id"):
                    if k in t and str(t[k]).strip():
                        out.append(str(t[k]).strip()); break
            else:
                s = str(t).strip().strip("'").strip('"')
                if s: out.append(s)
        return out
    if x is None or (isinstance(x, float) and np.isnan(x)): return []
    s = str(x).strip()
    # JSON?
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("{") and s.endswith("}")):
        try:
            obj = json.loads(s); return parse_chunk_ids(obj)
        except Exception: pass
    # strip brackets
    if s.startswith("[") and s.endswith("]"): s = s[1:-1]
    parts = [p.strip().strip("'").strip('"') for p in s.split(",")] if "," in s else \
            [p.strip().strip("'").strip('"') for p in re.split(r"\s+", s) if p.strip()]
    if not parts:
        parts = re.findall(r"[^\s,\[\]]+", s)
    return [p for p in parts if p not in ("[","]")]

# 1) 读数据
DFu = pd.read_parquet(IN_UNIFIED)
M1  = pd.read_parquet(PATH_M1)
M3  = pd.read_parquet(PATH_M3)

DFu["__qkey__"] = DFu["question"].apply(norm_q)
M1["__qkey__"]  = M1["question"].apply(norm_q)
M3["__qkey__"]  = M3["question"].apply(norm_q)

# 2) 读 meta.jsonl，建立两个映射：
#    - map_full:  完整 chunk_id（包含 paper_id::…） → text
#    - map_right: 右半段（去掉 paper_id::）         → 第一次出现的 text（退路）
meta = pd.read_json(META_JSONL, lines=True)
need = ["paper_id","chunk_id","text"]
assert all(c in meta.columns for c in need), f"meta.jsonl 缺列：{list(meta.columns)}"
meta = meta.dropna(subset=["chunk_id","text"]).drop_duplicates(subset=["paper_id","chunk_id"], keep="first")
meta["chunk_id"] = meta["chunk_id"].astype(str).str.strip()
meta["paper_id"] = meta["paper_id"].astype(str).str.strip()

map_full = dict(zip(meta["chunk_id"], meta["text"].astype(str)))  # 主映射
meta["right"] = meta["chunk_id"].str.split("::", n=1).str[-1]
map_right = meta.drop_duplicates(subset=["right"]).set_index("right")["text"].astype(str).to_dict()  # 退路

print(f"map_full size: {len(map_full):,} | map_right size: {len(map_right):,}")

# 3) 解析 M1 的检索 ID，并做映射（先 full，再 right）
M1["retrieved_chunks_parsed"] = M1["retrieved_chunks"].apply(parse_chunk_ids)

def ids_to_texts_fixed(ids: list):
    if not isinstance(ids, list) or not ids: return [], []
    out, miss = [], []
    for tok in ids:
        t0 = str(tok).strip().strip("'").strip('"')
        # 先用完整键直接命中
        t = map_full.get(t0)
        if t is None and "::" in t0:
            right = t0.split("::",1)[1]
            t = map_full.get(right) or map_right.get(right)  # 可覆盖右半段直接成键的情况
        elif t is None:
            t = map_right.get(t0)
        if t: out.append(t)
        else: miss.append(t0)
    return out, miss

mm = M1["retrieved_chunks_parsed"].apply(ids_to_texts_fixed)
M1["retrieved_ctx_text"] = [x[0] for x in mm]
M1["__miss_tokens__"]    = [x[1] for x in mm]

total_ids   = int(sum(len(v) for v in M1["retrieved_chunks_parsed"]))
matched_ids = int(sum(len(v) for v in M1["retrieved_ctx_text"]))
rows_nonempty = int(M1["retrieved_ctx_text"].apply(lambda x: isinstance(x,list) and len(x)>0).sum())
print(f"mapped rows with non-empty ctx: {rows_nonempty}/{len(M1)} | matched ids: {matched_ids}/{total_ids}")

# 打印几条未命中样例帮助核对
miss_flat = [z for lst in M1["__miss_tokens__"] if isinstance(lst, list) for z in lst]
print("missed token samples (<=12):", miss_flat[:12])

# 4) 合并回统一表（M1 用文本，M3 用自身 contexts）
def first_nonempty_list(series: pd.Series):
    for v in series:
        if isinstance(v, (list, tuple)) and len(v) > 0:
            return list(v)
    return []

M3["contexts"] = M3["contexts"].apply(lambda v: v if isinstance(v, list) else parse_chunk_ids(v))

M1_map = M1.groupby("__qkey__")["retrieved_ctx_text"].apply(first_nonempty_list)
M3_map = M3.groupby("__qkey__")["contexts"].apply(first_nonempty_list)

if "retrieved_ctx" not in DFu.columns:
    DFu["retrieved_ctx"] = [[] for _ in range(len(DFu))]
else:
    DFu["retrieved_ctx"] = DFu["retrieved_ctx"].apply(lambda v: v if isinstance(v, list) else [])

is_m1 = DFu["model"].astype(str).str.upper().eq("M1")
is_m3 = DFu["model"].astype(str).str.upper().eq("M3")

DF_out = DFu.copy()
DF_out.loc[is_m1, "retrieved_ctx"] = DF_out.loc[is_m1, "__qkey__"].map(M1_map).apply(lambda v: v if isinstance(v, list) else [])
DF_out.loc[is_m3, "retrieved_ctx"] = DF_out.loc[is_m3, "__qkey__"].map(M3_map).apply(lambda v: v if isinstance(v, list) else [])
DF_out["has_retrieval"] = DF_out["retrieved_ctx"].apply(lambda x: isinstance(x, list) and len(x)>0)
DF_out.drop(columns=["__qkey__"], inplace=True)

OUT_UNIFIED.parent.mkdir(parents=True, exist_ok=True)
DF_out.to_parquet(OUT_UNIFIED, index=False)
print("💾 wrote:", OUT_UNIFIED)

print("\nNon-empty retrieved_ctx by model:")
print(DF_out.assign(nonempty=DF_out["retrieved_ctx"].apply(lambda x: int(isinstance(x,list) and len(x)>0))) \
      .groupby("model")["nonempty"].sum())

# 看 2 条成功映射的文本（如果有）
ok_rows = M1[M1["retrieved_ctx_text"].apply(lambda x: isinstance(x,list) and len(x)>0)]
if len(ok_rows) > 0:
    print("\n✅ 示例（映射后的第一段文本预览）")
    r0 = ok_rows.iloc[0]
    print(r0["retrieved_chunks_parsed"][:3], "→")
    print((r0["retrieved_ctx_text"][0] or "")[:200].replace("\n"," "))
